# Modèle de régression linéaire multiple pour la prédiction des prix de l'immobilier 

## Importation des bibliothèques

In [1]:
import pickle
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.cluster import KMeans
from sklearn.preprocessing import PolynomialFeatures

## Utilisation de pickle pour appeler le jeu de données transformé

In [2]:
# Chemin relatif depuis le notebook
with open("df_transformed.pkl", "rb") as f:
    df = pickle.load(f)

In [3]:
X = df.drop(columns=["prix","id","date","zipcode","lat",'long',
       'm2_jardin', 'm2_soussol'])  # Variables explicatives
y = df["prix"]  # Target

In [4]:
# Train/test avec 20% pour le test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Mise en place du pipeline

In [5]:
# Pipeline : permet d'imputer les valeurs manquantes avec la moyenne, standardise les valeurs (moyenne 0 e-t 1), sélection des 
# variables les plus pertinentes et regression linéaire 
pipeline = Pipeline([
    ("poly", PolynomialFeatures(2)),  # Transformation polynomiale (degré 2)
    ("imputer", SimpleImputer(strategy="mean")),  # Imputation des valeurs manquantes
    ("scaler", StandardScaler()),  # Standardisation
    ("feature_selection", SelectKBest(score_func=f_regression, k=5)),  # Sélection des meilleures features
    ("model", LinearRegression())  # Régression linéaire
])

## Entraînement du modèle simple

In [6]:
# Entraînement 
pipeline.fit(X_train, y_train)

# Prédictions
y_pred = pipeline.predict(X_test)

# Calcul du RMSE comme demandé dans le sujet et r2 car parlant 
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"RMSE du modèle : {rmse:.2f}")
print(f"R² du modèle : {r2:.3f}")

RMSE du modèle : 164283.09
R² du modèle : 0.778


## Optimisation des hyperparamètres du modèle

In [7]:
# GridSearchCV pour optimiser la sélection de variables et le modèle
param_grid = {
    "feature_selection__k": [3, 5, 7, 9, 11, 13, 15, 17],  # Tester différentes sélections de variables
    "model__fit_intercept": [True, False] # Ajout de biais ou non 
}

# Recherche du RMSE minimum avec validation croisée 
grid_search = GridSearchCV(pipeline, param_grid, cv=10, scoring="neg_root_mean_squared_error", n_jobs=-1)
grid_search.fit(X_train, y_train)

# Meilleurs paramètres et scores optimisés
best_rmse = -grid_search.best_score_  #  Inversion car GridSearchCV retourne une valeur négative
best_r2 = grid_search.best_estimator_.score(X_test, y_test)  # Score R² du meilleur modèle

print(f"Meilleurs paramètres : {grid_search.best_params_}")
print(f"Meilleur RMSE en validation croisée : {best_rmse:.2f}")
print(f"R² du meilleur modèle : {best_r2:.3f}")

Meilleurs paramètres : {'feature_selection__k': 17, 'model__fit_intercept': True}
Meilleur RMSE en validation croisée : 147646.14
R² du meilleur modèle : 0.831
